In [12]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

In [13]:
from models import Ride, ride_deserializer

In [14]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer= ride_deserializer
)

In [15]:
next(consumer)

ConsumerRecord(topic='rides', partition=0, leader_epoch=2, offset=77, timestamp=1773335737968, timestamp_type=0, key=None, value=Ride(PULocationID=68, DOLocationID=265, trip_distance=4.31, total_amount=90.81, tpep_pickup_datetime=1761958059000), headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=126, serialized_header_size=-1)

In [16]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=90, DO=141, distance=3.92, amount=$42.42, pickup=2025-11-01 00:14:46
Received: PU=162, DO=173, distance=7.98, amount=$50.25, pickup=2025-11-01 00:54:20
Received: PU=238, DO=48, distance=3.03, amount=$35.44, pickup=2025-11-01 00:15:12
Received: PU=141, DO=140, distance=0.54, amount=$13.86, pickup=2025-11-01 00:19:04
Received: PU=237, DO=239, distance=1.46, amount=$20.50, pickup=2025-11-01 00:30:32
Received: PU=229, DO=48, distance=1.8, amount=$26.45, pickup=2025-11-01 00:38:41
Received: PU=48, DO=233, distance=0.91, amount=$17.85, pickup=2025-11-01 00:16:50
Received: PU=233, DO=113, distance=1.61, amount=$27.30, pickup=2025-11-01 00:33:49
Received: PU=79, DO=137, distance=1.33, amount=$19.25, pickup=2025-11-01 00:34:57
Received: PU=132, DO=265, distance=45.7, amount=$284.39, pickup=2025-11-01 00:24:16

... received 10 messages so far (stopping after 10 for demo)


In [17]:
#message = next(consumer)

In [18]:
message.value

Ride(PULocationID=132, DOLocationID=265, trip_distance=45.7, total_amount=284.39, tpep_pickup_datetime=1761956656000)

In [19]:
for message in consumer:
    print(message.value)

In [20]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer= ride_deserializer
)

In [21]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [22]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...


Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
Inserted 1600 rows...
Inserted 1700 rows...
Inserted 1800 rows...
Inserted 1900 rows...
Inserted 2000 rows...


KeyboardInterrupt: 